In [1]:
def predict_dropout1():
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    train['test_1_num']    = train['test 1'].str.upper().map(GRADE_MAP)
    train['forum_Q_s1']    = train['forum Q'] / 4
    train['forum_A_s1']    = train['forum A'] / 4
    train['oh_s1']         = train['office hour visits'] / 4
    train['session_total'] = train['session 1'].fillna(0) + train['session 2'].fillna(0)
    train['grade_mean']    = train['test_1_num']
    train['y']             = train['dropout'].map({'Y': 1, 'N': 0})

    feats = ['external_flag', 'year_num', 'session 1', 'session 2',
             'test_1_num', 'forum_Q_s1', 'forum_A_s1', 'oh_s1',
             'session_total', 'grade_mean']

    imp    = SimpleImputer(strategy='median')
    X_tr   = imp.fit_transform(train[feats].values)
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    y_tr   = train['y'].values

    model = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5, random_state=42)
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout1.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    test['test_1_num']    = test['test 1'].str.upper().map(GRADE_MAP)
    test['forum_Q_s1']    = test['forum Q'] / 4
    test['forum_A_s1']    = test['forum A'] / 4
    test['oh_s1']         = test['office hour visits'] / 4
    test['session_total'] = test['session 1'].fillna(0) + test['session 2'].fillna(0)
    test['grade_mean']    = test['test_1_num']

    X_te   = scaler.transform(imp.transform(test[feats].values))
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [2]:
def predict_dropout2():
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.impute import SimpleImputer

    GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2']:
        train[col.replace(' ', '_') + '_num'] = train[col].str.upper().map(GRADE_MAP)
    train['forum_Q_s2']    = train['forum Q'] / 2
    train['forum_A_s2']    = train['forum A'] / 2
    train['oh_s2']         = train['office hour visits'] / 2
    train['session_total'] = (train['session 1'].fillna(0) + train['session 2'].fillna(0) +
                              train['session 3'].fillna(0) + train['session 4'].fillna(0))
    train['grade_mean']    = train[['test_1_num', 'test_2_num']].mean(axis=1)
    train['grade_delta']   = train['test_2_num'] - train['test_1_num']
    train['y']             = train['dropout'].map({'Y': 1, 'N': 0})

    train = train[train['session 3'].notna() | train['test 2'].notna()]

    feats = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4',
             'test_1_num', 'test_2_num', 'forum_Q_s2', 'forum_A_s2', 'oh_s2',
             'session_total', 'grade_mean', 'grade_delta']

    imp  = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(train[feats].values)
    y_tr = train['y'].values

    model = RandomForestClassifier(n_estimators=300, max_depth=6,
                                   class_weight='balanced_subsample',
                                   random_state=42, n_jobs=-1)
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout2.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2']:
        test[col.replace(' ', '_') + '_num'] = test[col].str.upper().map(GRADE_MAP)
    test['forum_Q_s2']    = test['forum Q'] / 2
    test['forum_A_s2']    = test['forum A'] / 2
    test['oh_s2']         = test['office hour visits'] / 2
    test['session_total'] = (test['session 1'].fillna(0) + test['session 2'].fillna(0) +
                             test['session 3'].fillna(0) + test['session 4'].fillna(0))
    test['grade_mean']    = test[['test_1_num', 'test_2_num']].mean(axis=1)
    test['grade_delta']   = test['test_2_num'] - test['test_1_num']

    X_te   = imp.transform(test[feats].values)
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [3]:
def predict_dropout3():
    import pandas as pd
    import numpy as np
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.impute import SimpleImputer

    GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2', 'test 3']:
        train[col.replace(' ', '_') + '_num'] = train[col].str.upper().map(GRADE_MAP)
    train['forum_Q_s3']    = train['forum Q'] * 0.75
    train['forum_A_s3']    = train['forum A'] * 0.75
    train['oh_s3']         = train['office hour visits'] * 0.75
    train['session_total'] = (train['session 1'].fillna(0) + train['session 2'].fillna(0) +
                              train['session 3'].fillna(0) + train['session 4'].fillna(0) +
                              train['session 5'].fillna(0))
    train['grade_mean']    = train[['test_1_num', 'test_2_num', 'test_3_num']].mean(axis=1)
    train['grade_delta']   = train['test_3_num'] - train['test_1_num']
    train['y']             = train['dropout'].map({'Y': 1, 'N': 0})

    train = train[train['session 5'].notna() | train['test 3'].notna()]

    feats = ['external_flag', 'year_num',
             'session 1', 'session 2', 'session 3', 'session 4', 'session 5',
             'test_1_num', 'test_2_num', 'test_3_num',
             'forum_Q_s3', 'forum_A_s3', 'oh_s3',
             'session_total', 'grade_mean', 'grade_delta']

    imp  = SimpleImputer(strategy='median')
    X_tr = imp.fit_transform(train[feats].values)
    y_tr = train['y'].values

    model = HistGradientBoostingClassifier(max_iter=200, max_depth=5,
                                           class_weight='balanced', random_state=42)
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout3.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    for col in ['test 1', 'test 2', 'test 3']:
        test[col.replace(' ', '_') + '_num'] = test[col].str.upper().map(GRADE_MAP)
    test['forum_Q_s3']    = test['forum Q'] * 0.75
    test['forum_A_s3']    = test['forum A'] * 0.75
    test['oh_s3']         = test['office hour visits'] * 0.75
    test['session_total'] = (test['session 1'].fillna(0) + test['session 2'].fillna(0) +
                             test['session 3'].fillna(0) + test['session 4'].fillna(0) +
                             test['session 5'].fillna(0))
    test['grade_mean']    = test[['test_1_num', 'test_2_num', 'test_3_num']].mean(axis=1)
    test['grade_delta']   = test['test_3_num'] - test['test_1_num']

    X_te   = imp.transform(test[feats].values)
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [4]:
def predict_dropout4():
    import pandas as pd
    import numpy as np
    from sklearn.linear_model import LogisticRegression
    from sklearn.ensemble import (RandomForestClassifier,
                                  HistGradientBoostingClassifier,
                                  StackingClassifier)
    from sklearn.impute import SimpleImputer
    from sklearn.preprocessing import StandardScaler

    GRADE_MAP  = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
    YEAR_MAP   = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}
    GRADE_COLS = ['test 1', 'test 2', 'test 3', 'ind cw', 'group cw', 'final grade']

    train = pd.read_csv('ND26_dropout.csv')
    train['external_flag'] = train['External'].map({'Y': 1, 'N': 0})
    train['year_num']      = train['Year'].str.lower().map(YEAR_MAP)
    for col in GRADE_COLS:
        train[col.replace(' ', '_') + '_num'] = train[col].str.upper().map(GRADE_MAP)
    train['forum_Q_s4']    = train['forum Q']
    train['forum_A_s4']    = train['forum A']
    train['oh_s4']         = train['office hour visits']
    train['session_total'] = (train['session 1'].fillna(0) + train['session 2'].fillna(0) +
                              train['session 3'].fillna(0) + train['session 4'].fillna(0) +
                              train['session 5'].fillna(0) + train['session 6'].fillna(0))
    num_grade_cols = ['test_1_num', 'test_2_num', 'test_3_num',
                      'ind_cw_num', 'group_cw_num', 'final_grade_num']
    train['grade_mean']  = train[num_grade_cols].mean(axis=1)
    train['grade_delta'] = train['final_grade_num'] - train['test_1_num']
    train['y']           = train['dropout'].map({'Y': 1, 'N': 0})

    train = train[train['session 6'].notna() | train['ind cw'].notna()]

    feats = ['external_flag', 'year_num',
             'session 1', 'session 2', 'session 3', 'session 4', 'session 5', 'session 6',
             'test_1_num', 'test_2_num', 'test_3_num',
             'ind_cw_num', 'group_cw_num', 'final_grade_num',
             'forum_Q_s4', 'forum_A_s4', 'oh_s4',
             'session_total', 'grade_mean', 'grade_delta']

    imp    = SimpleImputer(strategy='median')
    X_tr   = imp.fit_transform(train[feats].values)
    scaler = StandardScaler()
    X_tr   = scaler.fit_transform(X_tr)
    y_tr   = train['y'].values

    model = StackingClassifier(
        estimators=[
            ('lr',  LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced')),
            ('rf',  RandomForestClassifier(n_estimators=200, random_state=42,
                                           class_weight='balanced_subsample', n_jobs=-1)),
            ('hgb', HistGradientBoostingClassifier(max_iter=150, random_state=42)),
        ],
        final_estimator=LogisticRegression(C=1.0),
        passthrough=False, cv=3
    )
    model.fit(X_tr, y_tr)

    test = pd.read_csv('./data/entry_dropout4.csv')
    test['external_flag'] = test['External'].map({'Y': 1, 'N': 0})
    test['year_num']      = test['Year'].str.lower().map(YEAR_MAP)
    for col in GRADE_COLS:
        test[col.replace(' ', '_') + '_num'] = test[col].str.upper().map(GRADE_MAP)
    test['forum_Q_s4']    = test['forum Q']
    test['forum_A_s4']    = test['forum A']
    test['oh_s4']         = test['office hour visits']
    test['session_total'] = (test['session 1'].fillna(0) + test['session 2'].fillna(0) +
                             test['session 3'].fillna(0) + test['session 4'].fillna(0) +
                             test['session 5'].fillna(0) + test['session 6'].fillna(0))
    test['grade_mean']  = test[num_grade_cols].mean(axis=1)
    test['grade_delta'] = test['final_grade_num'] - test['test_1_num']

    X_te   = scaler.transform(imp.transform(test[feats].values))
    probas = model.predict_proba(X_te)[:, 1]
    thresh = np.median(probas)
    return ['Y' if p >= thresh else 'N' for p in probas]

In [5]:
print("Stage 1:", predict_dropout1())
print("Stage 2:", predict_dropout2())
print("Stage 3:", predict_dropout3())
print("Stage 4:", predict_dropout4())

Stage 1: ['Y', 'N', 'N', 'Y', 'N', 'Y', 'Y', 'N', 'N', 'Y']
Stage 2: ['Y', 'Y', 'N', 'N', 'N', 'N', 'Y', 'Y', 'Y', 'N']
Stage 3: ['Y', 'Y', 'Y', 'N', 'N', 'Y', 'N', 'Y', 'N', 'N']
Stage 4: ['N', 'Y', 'N', 'N', 'Y', 'N', 'Y', 'Y', 'Y', 'N']


In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (RandomForestClassifier,
                              HistGradientBoostingClassifier,
                              StackingClassifier)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

GRADE_MAP = {'F': 0, 'E': 1, 'D': 2, 'C': 3, 'B': 4, 'A': 5}
YEAR_MAP  = {'first': 1, 'second': 2, 'third': 3, 'fourth': 4}

train_full = pd.read_csv('ND26_dropout.csv')
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def _base(df):
    df = df.copy()
    df['external_flag'] = df['External'].map({'Y': 1, 'N': 0})
    df['year_num']      = df['Year'].str.lower().map(YEAR_MAP)
    df['y']             = df['dropout'].map({'Y': 1, 'N': 0})
    return df

# ── Stage 1 ──────────────────────────────────────────────────────────────────
t1 = _base(train_full)
t1['test_1_num']    = t1['test 1'].str.upper().map(GRADE_MAP)
t1['forum_Q_s1']    = t1['forum Q'] / 4
t1['forum_A_s1']    = t1['forum A'] / 4
t1['oh_s1']         = t1['office hour visits'] / 4
t1['session_total'] = t1['session 1'].fillna(0) + t1['session 2'].fillna(0)
t1['grade_mean']    = t1['test_1_num']

feats1 = ['external_flag', 'year_num', 'session 1', 'session 2',
          'test_1_num', 'forum_Q_s1', 'forum_A_s1', 'oh_s1',
          'session_total', 'grade_mean']

imp1 = SimpleImputer(strategy='median')
X1 = StandardScaler().fit_transform(imp1.fit_transform(t1[feats1].values))
y1 = t1['y'].values

m1 = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5, random_state=42)
y1_pred = cross_val_predict(m1, X1, y1, cv=cv)

print("=" * 60)
print("STAGE 1 — Logistic Regression  (after session 2 / test 1)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y1, y1_pred):.4f}\n")
print(classification_report(y1, y1_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))

# ── Stage 2 ──────────────────────────────────────────────────────────────────
t2 = _base(train_full)
for col in ['test 1', 'test 2']:
    t2[col.replace(' ', '_') + '_num'] = t2[col].str.upper().map(GRADE_MAP)
t2['forum_Q_s2']    = t2['forum Q'] / 2
t2['forum_A_s2']    = t2['forum A'] / 2
t2['oh_s2']         = t2['office hour visits'] / 2
t2['session_total'] = (t2['session 1'].fillna(0) + t2['session 2'].fillna(0) +
                       t2['session 3'].fillna(0) + t2['session 4'].fillna(0))
t2['grade_mean']    = t2[['test_1_num', 'test_2_num']].mean(axis=1)
t2['grade_delta']   = t2['test_2_num'] - t2['test_1_num']
t2 = t2[t2['session 3'].notna() | t2['test 2'].notna()]

feats2 = ['external_flag', 'year_num', 'session 1', 'session 2', 'session 3', 'session 4',
          'test_1_num', 'test_2_num', 'forum_Q_s2', 'forum_A_s2', 'oh_s2',
          'session_total', 'grade_mean', 'grade_delta']

imp2 = SimpleImputer(strategy='median')
X2   = imp2.fit_transform(t2[feats2].values)
y2   = t2['y'].values

m2 = RandomForestClassifier(n_estimators=300, max_depth=6,
                             class_weight='balanced_subsample', random_state=42, n_jobs=-1)
y2_pred = cross_val_predict(m2, X2, y2, cv=cv)

print("=" * 60)
print("STAGE 2 — Random Forest        (after session 4 / test 2)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y2, y2_pred):.4f}\n")
print(classification_report(y2, y2_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))

# ── Stage 3 ──────────────────────────────────────────────────────────────────
t3 = _base(train_full)
for col in ['test 1', 'test 2', 'test 3']:
    t3[col.replace(' ', '_') + '_num'] = t3[col].str.upper().map(GRADE_MAP)
t3['forum_Q_s3']    = t3['forum Q'] * 0.75
t3['forum_A_s3']    = t3['forum A'] * 0.75
t3['oh_s3']         = t3['office hour visits'] * 0.75
t3['session_total'] = (t3['session 1'].fillna(0) + t3['session 2'].fillna(0) +
                       t3['session 3'].fillna(0) + t3['session 4'].fillna(0) +
                       t3['session 5'].fillna(0))
t3['grade_mean']    = t3[['test_1_num', 'test_2_num', 'test_3_num']].mean(axis=1)
t3['grade_delta']   = t3['test_3_num'] - t3['test_1_num']
t3 = t3[t3['session 5'].notna() | t3['test 3'].notna()]

feats3 = ['external_flag', 'year_num',
          'session 1', 'session 2', 'session 3', 'session 4', 'session 5',
          'test_1_num', 'test_2_num', 'test_3_num',
          'forum_Q_s3', 'forum_A_s3', 'oh_s3',
          'session_total', 'grade_mean', 'grade_delta']

imp3 = SimpleImputer(strategy='median')
X3   = imp3.fit_transform(t3[feats3].values)
y3   = t3['y'].values

m3 = HistGradientBoostingClassifier(max_iter=200, max_depth=5,
                                    class_weight='balanced', random_state=42)
y3_pred = cross_val_predict(m3, X3, y3, cv=cv)

print("=" * 60)
print("STAGE 3 — HistGradientBoosting  (after session 5 / test 3)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y3, y3_pred):.4f}\n")
print(classification_report(y3, y3_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))

# ── Stage 4 ──────────────────────────────────────────────────────────────────
GRADE_COLS4    = ['test 1', 'test 2', 'test 3', 'ind cw', 'group cw', 'final grade']
num_grade_cols = ['test_1_num', 'test_2_num', 'test_3_num',
                  'ind_cw_num', 'group_cw_num', 'final_grade_num']

t4 = _base(train_full)
for col in GRADE_COLS4:
    t4[col.replace(' ', '_') + '_num'] = t4[col].str.upper().map(GRADE_MAP)
t4['forum_Q_s4']    = t4['forum Q']
t4['forum_A_s4']    = t4['forum A']
t4['oh_s4']         = t4['office hour visits']
t4['session_total'] = (t4['session 1'].fillna(0) + t4['session 2'].fillna(0) +
                       t4['session 3'].fillna(0) + t4['session 4'].fillna(0) +
                       t4['session 5'].fillna(0) + t4['session 6'].fillna(0))
t4['grade_mean']    = t4[num_grade_cols].mean(axis=1)
t4['grade_delta']   = t4['final_grade_num'] - t4['test_1_num']
t4 = t4[t4['session 6'].notna() | t4['ind cw'].notna()]

feats4 = ['external_flag', 'year_num',
          'session 1', 'session 2', 'session 3', 'session 4', 'session 5', 'session 6',
          'test_1_num', 'test_2_num', 'test_3_num',
          'ind_cw_num', 'group_cw_num', 'final_grade_num',
          'forum_Q_s4', 'forum_A_s4', 'oh_s4',
          'session_total', 'grade_mean', 'grade_delta']

imp4    = SimpleImputer(strategy='median')
X4      = StandardScaler().fit_transform(imp4.fit_transform(t4[feats4].values))
y4      = t4['y'].values

m4 = StackingClassifier(
    estimators=[
        ('lr',  LogisticRegression(max_iter=1000, C=0.5, class_weight='balanced')),
        ('rf',  RandomForestClassifier(n_estimators=200, random_state=42,
                                       class_weight='balanced_subsample', n_jobs=-1)),
        ('hgb', HistGradientBoostingClassifier(max_iter=150, random_state=42)),
    ],
    final_estimator=LogisticRegression(C=1.0),
    passthrough=False, cv=3
)
y4_pred = cross_val_predict(m4, X4, y4, cv=cv)

print("=" * 60)
print("STAGE 4 — Stacking Ensemble     (end-of-year, all features)")
print("=" * 60)
print(f"Accuracy: {accuracy_score(y4, y4_pred):.4f}\n")
print(classification_report(y4, y4_pred, target_names=['No Dropout (N)', 'Dropout (Y)']))


STAGE 1 — Logistic Regression  (after session 2 / test 1)
Accuracy: 0.8852

                precision    recall  f1-score   support

No Dropout (N)       0.91      0.87      0.89     13805
   Dropout (Y)       0.85      0.91      0.88     11944

      accuracy                           0.89     25749
     macro avg       0.88      0.89      0.89     25749
  weighted avg       0.89      0.89      0.89     25749

STAGE 2 — Random Forest        (after session 4 / test 2)
Accuracy: 0.9372

                precision    recall  f1-score   support

No Dropout (N)       0.96      0.93      0.95     13805
   Dropout (Y)       0.90      0.95      0.92      9004

      accuracy                           0.94     22809
     macro avg       0.93      0.94      0.93     22809
  weighted avg       0.94      0.94      0.94     22809

STAGE 3 — HistGradientBoosting  (after session 5 / test 3)
Accuracy: 0.9670

                precision    recall  f1-score   support

No Dropout (N)       0.99      0.97 